# SheraTutor — Unsloth Qwen-2.5-7B-Instruct Fine-Tuning Pipeline

This notebook fine-tunes **Qwen 2.5 7B** (or 3B) using **Unsloth** on **Kaggle** (or **Google Colab**) with a free Tesla T4 GPU.
It uses QLoRA 4-bit precision, Triton kernels (2x-5x faster training, 70% less VRAM), ChatML formatting, and response-only loss masking.

### Kaggle Settings:
1. **Accelerator**: `GPU T4 x2` or `GPU T4 x1` (**Do NOT select P100**)
2. **Internet**: `ON`
3. **Secrets**: Add `HF_TOKEN` (Hugging Face write token)

In [ ]:
%%capture
# 1. Install Unsloth and Dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes datasets transformers

In [ ]:
import os
import torch

# 2. Verify GPU Architecture
assert torch.cuda.is_available(), "Please enable GPU Accelerator (T4) in Notebook Settings!"
cap = torch.cuda.get_device_capability()
print(f"[*] Active GPU: {torch.cuda.get_device_name(0)} (Compute Capability {cap[0]}.{cap[1]})")
if cap[0] < 7:
    raise SystemError("GPU capability is < 7.0 (P100). Please switch to Tesla T4 in Kaggle Settings!")

In [ ]:
# 3. Hugging Face Authentication
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("[*] Retrieved HF_TOKEN from Kaggle Secrets.")
except Exception:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("[*] Retrieved HF_TOKEN from Colab Secrets.")
    except Exception:
        print("[!] No secret manager found. Please ensure HF_TOKEN is defined if pushing to Hub.")

HF_USERNAME = "syed181"  # Set your Hugging Face username
OUTPUT_MODEL_NAME = f"{HF_USERNAME}/sheratutor-qwen2.5-7b"

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, DataCollatorForSeq2Seq
from transformers import TrainingArguments

# 4. Load Qwen-2.5-7B-Instruct in 4-bit
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 5. Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 6. Configure Chat Template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# 7. Load & Format Dataset
dataset_url = "https://raw.githubusercontent.com/unslothai/datasets/main/gemma-prompt.json" # Or upload sheratutor_supabase_rag_dataset.jsonl
# When running locally or with attached Kaggle dataset:
dataset_file = "sheratutor_supabase_rag_dataset.jsonl"
if not os.path.exists(dataset_file):
    # Fallback search paths
    for p in ["../dataset/sheratutor_supabase_rag_dataset.jsonl", "/kaggle/input/sheratutor-dataset/sheratutor_supabase_rag_dataset.jsonl"]:
        if os.path.exists(p):
            dataset_file = p
            break

print(f"Loading dataset from: {dataset_file}")
raw_dataset = load_dataset("json", data_files=dataset_file, split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = raw_dataset.map(formatting_prompts_func, batched=True)
print(f"Total training examples: {len(dataset)}")

In [ ]:
# 8. Initialize Trainer with Response-Only Loss Masking
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# 9. Execute Training
trainer_stats = trainer.train()

In [ ]:
# 10. Fast Inference Test
FastLanguageModel.for_inference(model)

test_prompt = [
    {"role": "system", "content": "তুমি সেরাটিউটর — এসএসসি পর্যায়ের বিজ্ঞান শিক্ষক। সংকেতের মাধ্যমে সাহায্য করো।"},
    {"role": "user", "content": "স্যার, স্থির অবস্থান থেকে চলা শুরু করলে সমীকরণ কেমন হবে?"}
]
inputs = tokenizer.apply_chat_template(test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.3)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 11. Export to Hugging Face Hub (Merged 16-bit for vLLM & GGUF for Ollama)
if HF_TOKEN:
    print(f"Pushing 16-bit merged model to: {OUTPUT_MODEL_NAME}...")
    model.push_to_hub_merged(OUTPUT_MODEL_NAME, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    
    print(f"Pushing GGUF (Q4_K_M) to: {OUTPUT_MODEL_NAME}-gguf...")
    model.push_to_hub_gguf(f"{OUTPUT_MODEL_NAME}-gguf", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
    print("[SUCCESS] Models published to Hugging Face Hub!")
else:
    print("Saving locally to /kaggle/working/sheratutor_model...")
    model.save_pretrained_merged("/kaggle/working/sheratutor_model", tokenizer, save_method="merged_16bit")
    model.save_pretrained_gguf("/kaggle/working/sheratutor_gguf", tokenizer, quantization_method="q4_k_m")